In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter

In [13]:
# Set seed for reproducibility
torch.manual_seed(42)

# 1. Dataset (Slightly different sentences)
sentences = [
    "great performance by actors",
    "i loved the ending",
    "awesome script and direction",
    "worst execution ever seen",
    "i disliked the plot",
    "boring scenes and terrible"
]
# 1 = Positive, 0 = Negative
labels = torch.tensor([1.0, 1.0, 1.0, 0.0, 0.0, 0.0], dtype=torch.float32) # Stricly a tensor if we are using torch

In [14]:
# 2. Manual Vectorization (Tokenization + Vocabulary)
# Unlike keras we have to do this step manually , but also giving us more flexibility

# Split sentences into words
tokenized_sentences = [s.split() for s in sentences]

# Build vocabulary
word_counts = Counter(word for s in tokenized_sentences for word in s)
vocab = {word: i + 2 for i, (word, _) in enumerate(word_counts.items())}

# Dummy variables
vocab["<PAD>"] = 0
vocab["<OOV>"] = 1  # Out of vocabulary

In [15]:
# Numericalize and Pad sequences to max_length = 4
max_length = 4
X_list = []
for tokens in tokenized_sentences:
    # Convert words to IDs
    ids = [vocab.get(token, 1) for token in tokens]
    # Pad or truncate
    if len(ids) < max_length:
        ids += [0] * (max_length - len(ids))
    else:
        ids = ids[:max_length]
    X_list.append(ids)

In [16]:
X_train = torch.tensor(X_list, dtype=torch.long) # Create the training set

In [17]:
# 3. Define the PyTorch RNN Architecture
class TextRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(TextRNN, self).__init__()
        # 1st Layer: Embedding (Keras equivalent)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # 2nd Layer: Vanilla RNN
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        # 3rd Layer: Fully Connected Output Layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text) # shape: [batch_size, seq_len, embedding_dim]

        # output: all hidden states, hidden: final hidden state of the last time step
        output, hidden = self.rnn(embedded)

        # Take the final hidden state [1, batch_size, hidden_dim] and squeeze it
        final_state = hidden.squeeze(0)

        # Pass through dense layer and apply Sigmoid via Loss function later
        return self.fc(final_state)

In [18]:
# Initialize network components
vocab_size = len(vocab)
model = TextRNN(vocab_size=vocab_size, embedding_dim=8, hidden_dim=16)
criterion = nn.BCEWithLogitsLoss() # Combines Sigmoid + Binary Crossentropy
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [19]:
# 4. Training Loop
epochs = 5
batch_size = 2

print("Training PyTorch RNN...")
for epoch in range(epochs):
    epoch_loss = 0
    # Process data in manual batches
    for i in range(0, len(X_train), batch_size):
        x_batch = X_train[i:i+batch_size]
        y_batch = labels[i:i+batch_size].unsqueeze(1) # Match output shape [batch, 1]

        # Forward pass
        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss / (len(X_train)/batch_size):.4f}")

Training PyTorch RNN...
Epoch 1/5 - Loss: 0.7811
Epoch 2/5 - Loss: 0.6584
Epoch 3/5 - Loss: 0.5802
Epoch 4/5 - Loss: 0.5030
Epoch 5/5 - Loss: 0.4220


In [20]:
# Put the model in evaluation mode (turns off dropout/batchnorm if any)
model.eval()


TextRNN(
  (embedding): Embedding(23, 8, padding_idx=0)
  (rnn): RNN(8, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)

In [21]:
# Provide completely new phrases for testing
test_sentences = [
    "awesome performance",
    "terrible plot and boring"
]

print("\n--- Running Inference ---")

# Disable gradient calculations to save memory and speed up processing
with torch.no_grad():
    for sentence in test_sentences:
        # Preprocess the sentence exactly like the training data
        tokens = sentence.split()

        # Look up word IDs (use 1 for unknown <OOV> words)
        ids = [vocab.get(token, 1) for token in tokens]

        # Pad or truncate to max_length (4)
        if len(ids) < max_length:
            ids += [0] * (max_length - len(ids))
        else:
            ids = ids[:max_length]

        # Convert to a PyTorch tensor and add a batch dimension: [1, seq_len]
        input_tensor = torch.tensor([ids], dtype=torch.long)

        # Forward pass through the model to get raw logit scores
        logit_output = model(input_tensor)

        # Apply Sigmoid mathematically to convert logit to a 0-1 probability
        probability = torch.sigmoid(logit_output).item()

        # Determine final sentiment label based on a 0.5 threshold
        sentiment = "Positive" if probability >= 0.5 else "Negative"

        print(f"Text: '{sentence}'")
        print(f"↳ Raw Logit: {logit_output.item():.4f}")
        print(f"↳ Confidence Score: {probability * 100:.2f}%")
        print(f"↳ Predicted Label: {sentiment}\n")


--- Running Inference ---
Text: 'awesome performance'
↳ Raw Logit: 0.6997
↳ Confidence Score: 66.81%
↳ Predicted Label: Positive

Text: 'terrible plot and boring'
↳ Raw Logit: 0.3356
↳ Confidence Score: 58.31%
↳ Predicted Label: Positive

